In [5]:
def check_car_id(car_id: str) -> str:
    car_id = car_id.strip()

    replace = {
        'A': 'А', 'B': 'В', 'E': 'Е', 'K': 'К',
        'M': 'М', 'H': 'Н', 'O': 'О', 'P': 'Р',
        'C': 'С', 'T': 'Т', 'Y': 'У', 'X': 'Х'
    }

    normalized = ''
    for letter in car_id:
        if letter in replace:
            normalized += replace[letter]
        else:
            normalized += letter

    car_id = normalized

    if len(car_id) < 8 or len(car_id) > 9:
        return 'Номер не валиден.'

    first_letter = car_id[0]
    digits = car_id[1:4]
    last_letters = car_id[4:6]
    region = car_id[6:]

    allowed_letters = 'АВЕКМНОРСТУХ'

    if first_letter not in allowed_letters:
        return 'Номер не валиден.'

    if not digits.isdigit():
        return 'Номер не валиден.'

    if last_letters[0] not in allowed_letters or last_letters[1] not in allowed_letters:
        return 'Номер не валиден.'

    if not region.isdigit():
        return 'Номер не валиден.'

    if len(region) != 2 and len(region) != 3:
        return 'Номер не валиден.'

    car_number = first_letter + digits + last_letters

    return f'Номер {car_number} валиден. Регион: {region}.'

In [7]:
 check_car_id('А222ВС96')

'Номер А222ВС валиден. Регион: 96.'

In [11]:
check_car_id('А123ВЕ777')

'Номер А123ВЕ валиден. Регион: 777.'

In [13]:
check_car_id('АБ22ВВ193')

'Номер не валиден.'

In [18]:
import re
from datetime import datetime
 
LINE_RE = re.compile(
    r"""^[ \t]*
        (?P<ts>\d{4}-\d{2}-\d{2}[ ]\d{2}:\d{2}:\d{2})
        [ \t]+
        (?P<level>DEBUG|INFO|WARN|ERROR)
        (?:[ \t]+(?P<tail>.*?))?
        [ \t]*$""",
    re.MULTILINE | re.VERBOSE,
)
 
MSG_RE = re.compile(r"(?<!\S)msg=(?P<msg>.*?)(?=[ \t]+\w+=|$)")
KV_RE = re.compile(r"(?<!\S)(?P<key>user|action|ip|card)=(?P<val>\S*)")
 
_LABEL = r"[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?"
EMAIL_RE = re.compile(
    rf"[A-Za-z0-9_%+\-]+(?:\.[A-Za-z0-9_%+\-]+)*"
    rf"@(?:{_LABEL}\.)+[A-Za-z]{{2,}}"
)
_OCTET = r"(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)"
IPV4_RE = re.compile(rf"{_OCTET}(?:\.{_OCTET}){{3}}")
CARD_RE = re.compile(
    r"(?:\d{4}|X{4})-(?:\d{4}|X{4})-(?:\d{4}|X{4})-(?P<last4>\d{4})"
)
 
DUP_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+", re.IGNORECASE)
 
 
def dedupe_words(text: str) -> str:
    return DUP_WORD_RE.sub(r"\1", text)
 
 
def _valid_ts(ts: str) -> bool:
    try:
        datetime.strptime(ts, "%Y-%m-%d %H:%M:%S")
        return True
    except ValueError:
        return False
 
 
def parse_logs(text: str) -> list[dict]:
    result: list[dict] = []
    if not text or not text.strip():
        return result
 
    for m in LINE_RE.finditer(text.replace("\r\n", "\n")):
        ts = m.group("ts")
        if not _valid_ts(ts):
            continue
        tail = m.group("tail") or ""
 
        rec: dict = {"ts": ts, "level": m.group("level")}
        errors: list[str] = []
 
        msg_m = MSG_RE.search(tail)
        if msg_m:
            tail = tail[: msg_m.start()] + " " + tail[msg_m.end():]
 
        fields: dict[str, str] = {}
        for kv in KV_RE.finditer(tail):
            fields.setdefault(kv.group("key"), kv.group("val"))
 
        user = fields.get("user")
        rec["user"] = user or None
        if not user:
            errors.append("missing_user")
        elif not EMAIL_RE.fullmatch(user):
            errors.append("invalid_email")
 
        action = fields.get("action")
        rec["action"] = action or None
        if not action:
            errors.append("missing_action")
 
        ip = fields.get("ip")
        rec["ip"] = ip or None
        if not ip:
            errors.append("missing_ip")
        elif not IPV4_RE.fullmatch(ip):
            errors.append("invalid_ip")
 
        card = fields.get("card")
        rec["card"] = None
        if card is not None:
            cm = CARD_RE.fullmatch(card)
            if cm:
                rec["card"] = f"****-****-****-{cm.group('last4')}"
            else:
                errors.append("invalid_card")
 
        if msg_m:
            rec["msg"] = dedupe_words(msg_m.group("msg").strip().strip('"'))
 
        rec["errors"] = errors
        result.append(rec)
    return result

In [19]:
>>> parse_logs("2024-03-15 14:23:11 ERROR user=bad..email@ action=pay ip=999.1.1.1 card=4111-1111-1111-1111")
[{'ts': '2024-03-15 14:23:11', 'level': 'ERROR', 'user': 'bad..email@',
  'action': 'pay', 'ip': '999.1.1.1', 'card': '****-****-****-1111',
  'errors': ['invalid_email', 'invalid_ip']}]

[{'ts': '2024-03-15 14:23:11',
  'level': 'ERROR',
  'user': 'bad..email@',
  'action': 'pay',
  'ip': '999.1.1.1',
  'card': '****-****-****-1111',
  'errors': ['invalid_email', 'invalid_ip']}]